# Lab 1.1: Transformer Implementation from Scratch

## 🎯 Learning Objectives
By completing this lab, you will:
1. **Understand** the tensor operations in multi-head attention at a granular level
2. **Calculate** memory requirements for different model configurations
3. **Profile** computational complexity and identify bottlenecks
4. **Experiment** with architectural variations and measure their impact
5. **Connect** theoretical FLOP counts to actual measured performance

## 📊 Why This Matters
Implementing from scratch reveals insights that high-level APIs hide:
- Memory access patterns that determine real-world performance
- The actual cost of each operation (not just Big-O)
- Trade-offs in architectural decisions (heads, dimensions, etc.)
- Bottlenecks that optimization techniques target

## 🧪 Lab Structure
This lab has **5 parts**, each building on the previous:
1. **Basic Implementation**: Transformer layer components
2. **Memory Analysis**: Calculate and measure memory usage
3. **Performance Profiling**: Time operations and identify bottlenecks
4. **Architectural Experiments**: Vary parameters and observe effects
5. **Optimization Exploration**: Try simple optimizations

## 🔧 Prerequisites Check

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Optional, Tuple, Dict, List
import time
import psutil
import os
import matplotlib.pyplot as plt
from dataclasses import dataclass
import json

# Setup plotting
plt.style.use('seaborn-v0_8')
%matplotlib inline

# Check environment
print("=" * 60)
print("Environment Check")
print("=" * 60)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device = torch.cuda.get_device_properties(0)
    print(f"GPU: {device.name}")
    print(f"  Compute capability: {device.major}.{device.minor}")
    print(f"  Total memory: {device.total_memory / 1024**3:.1f} GB")
    print(f"  Multiprocessors: {device.multi_processor_count}")
    
    # Quick performance test
    a = torch.randn(4096, 4096, device='cuda')
    b = torch.randn(4096, 4096, device='cuda')
    
    torch.cuda.synchronize()
    start = time.time()
    c = torch.matmul(a, b)
    torch.cuda.synchronize()
    elapsed = time.time() - start
    
    flops = 2 * 4096**3 / elapsed
    print(f"  Matrix multiply performance: {flops / 1e12:.2f} TFLOPS")
    
    # Theoretical peak (approx)
    peak_tflops = device.multi_processor_count * device.clock_rate * 2 / 1e3
    print(f"  Theoretical peak (FP16): {peak_tflops:.1f} TFLOPS")
    print(f"  Achieved percentage: {flops / (peak_tflops * 1e12) * 100:.1f}%")
    
    del a, b, c
else:
    print("GPU not available. This lab will run on CPU but will be slow.")
    print("Consider using Google Colab for GPU access.")

# System info
print(f"\nCPU cores: {psutil.cpu_count()}")
mem = psutil.virtual_memory()
print(f"RAM: {mem.total / 1024**3:.1f} GB total, {mem.available / 1024**3:.1f} GB available")

# Part 1: Basic Implementation with Detailed Explanations

We'll implement each component with extensive comments explaining **why** each operation exists and **what** it costs.

## 1.1 Multi-Head Attention: The Heart of Transformers

In [ ]:
class EducationalMultiHeadAttention(nn.Module):
    """
    Multi-head attention with educational comments explaining each operation.
    
    Key concepts:
    1. **Projections**: Linear layers transform input to Q, K, V spaces
    2. **Heads**: Parallel attention mechanisms over different subspaces
    3. **Scaling**: 1/sqrt(d_k) prevents softmax saturation
    4. **Masking**: For padding and causal (autoregressive) attention
    5. **Softmax**: Converts scores to probabilities
    6. **Output projection**: Combines heads back to original dimension
    """
    
    def __init__(self, d_model: int = 512, num_heads: int = 8, dropout: float = 0.1, 
                 bias: bool = True, causal: bool = False):
        super().__init__()
        
        # Why assert divisibility? Each head needs equal dimension
        assert d_model % num_heads == 0, \
            f"d_model ({d_model}) must be divisible by num_heads ({num_heads})"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.causal = causal
        
        # Q, K, V projections: Could be fused for efficiency but separate for clarity
        # Each is d_model → d_model, so we maintain dimension
        self.q_proj = nn.Linear(d_model, d_model, bias=bias)
        self.k_proj = nn.Linear(d_model, d_model, bias=bias)
        self.v_proj = nn.Linear(d_model, d_model, bias=bias)
        
        # Output projection: Combines heads back to d_model
        self.out_proj = nn.Linear(d_model, d_model, bias=bias)
        
        # Dropout for regularization (not always used in inference)
        self.dropout = nn.Dropout(dropout)
        
        # Scaling factor: 1/sqrt(d_k) prevents extreme values before softmax
        # Without this, large d_k causes dot products to have large variance,
        # pushing softmax into regions with tiny gradients
        self.scale = self.head_dim ** -0.5
        
        # Causal mask for autoregressive generation (optional)
        if causal:
            self.register_buffer(
                "causal_mask",
                torch.triu(torch.full((1024, 1024), float('-inf')), diagonal=1)
            )
    
    def forward(self, 
                query: torch.Tensor, 
                key: torch.Tensor, 
                value: torch.Tensor,
                key_padding_mask: Optional[torch.Tensor] = None,
                need_weights: bool = False) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Forward pass with detailed comments on tensor shapes and operations.
        
        Args:
            query: [batch_size, query_seq_len, d_model]
            key: [batch_size, key_seq_len, d_model]
            value: [batch_size, value_seq_len, d_model]
            key_padding_mask: [batch_size, key_seq_len] (True for padding)
            need_weights: Whether to return attention weights
            
        Returns:
            output: [batch_size, query_seq_len, d_model]
            attn_weights: Optional [batch_size, num_heads, query_seq_len, key_seq_len]
        """
        batch_size, query_len, _ = query.shape
        _, key_len, _ = key.shape
        
        # ========== STEP 1: Project to Q, K, V spaces ==========
        # Linear transformations: O(nd²) FLOPs each
        Q = self.q_proj(query)  # [batch_size, query_len, d_model]
        K = self.k_proj(key)    # [batch_size, key_len, d_model]
        V = self.v_proj(value)  # [batch_size, key_len, d_model]
        
        # ========== STEP 2: Reshape for multi-head ==========
        # Reshape: [batch, seq_len, d_model] → [batch, seq_len, num_heads, head_dim]
        # Then transpose to: [batch, num_heads, seq_len, head_dim]
        # This puts heads in dimension 1 for parallel computation
        Q = Q.view(batch_size, query_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, key_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, key_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # ========== STEP 3: Compute attention scores ==========
        # Q @ K^T: [batch, heads, query_len, key_len]
        # Complexity: O(n²d) where n=seq_len, d=head_dim
        attn_scores = torch.matmul(Q, K.transpose(-2, -1))  # [batch, heads, query_len, key_len]
        
        # Scale to control variance (critical for training stability)
        attn_scores = attn_scores * self.scale
        
        # ========== STEP 4: Apply masks ==========
        if self.causal and query_len == key_len:
            # Causal mask: prevent attending to future tokens
            mask = self.causal_mask[:query_len, :key_len]
            attn_scores = attn_scores + mask.unsqueeze(0).unsqueeze(0)
        
        if key_padding_mask is not None:
            # Padding mask: prevent attending to padding tokens
            # Expand from [batch, key_len] to [batch, 1, 1, key_len]
            key_padding_mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_scores = attn_scores.masked_fill(key_padding_mask, float('-inf'))
        
        # ========== STEP 5: Softmax ==========
        # Softmax over last dimension (key_len)
        # Each query position gets probability distribution over keys
        attn_weights = F.softmax(attn_scores, dim=-1)  # [batch, heads, query_len, key_len]
        attn_weights = self.dropout(attn_weights)
        
        # ========== STEP 6: Apply attention to values ==========
        # attn_weights @ V: [batch, heads, query_len, head_dim]
        # Another O(n²d) operation
        context = torch.matmul(attn_weights, V)  # [batch, heads, query_len, head_dim]
        
        # ========== STEP 7: Concatenate heads ==========
        # Transpose back: [batch, heads, query_len, head_dim] → [batch, query_len, heads, head_dim]
        # Then reshape: [batch, query_len, d_model]
        context = context.transpose(1, 2).contiguous().view(batch_size, query_len, self.d_model)
        
        # ========== STEP 8: Output projection ==========
        # Final linear layer: O(nd²) FLOPs
        output = self.out_proj(context)  # [batch, query_len, d_model]
        
        return output, (attn_weights if need_weights else None)
    
    def flop_count(self, batch_size: int, query_len: int, key_len: int) -> Dict[str, int]:
        """Calculate theoretical FLOPs for this forward pass."""
        # Q/K/V projections: 3 × batch × seq × d_model × d_model × 2
        proj_flops = 3 * 2 * batch_size * query_len * self.d_model * self.d_model
        
        # QK^T: batch × heads × query_len × key_len × head_dim × 2
        qk_flops = 2 * batch_size * self.num_heads * query_len * key_len * self.head_dim
        
        # Softmax: approx 3 × batch × heads × query_len × key_len
        softmax_flops = 3 * batch_size * self.num_heads * query_len * key_len
        
        # Attention × V: same as QK^T
        av_flops = qk_flops
        
        # Output projection: same as one input projection
        out_flops = 2 * batch_size * query_len * self.d_model * self.d_model
        
        total = proj_flops + qk_flops + softmax_flops + av_flops + out_flops
        
        return {
            'projections': proj_flops,
            'qk_matmul': qk_flops,
            'softmax': softmax_flops,
            'av_matmul': av_flops,
            'output_proj': out_flops,
            'total': total
        }

## 1.2 Feed-Forward Network: The Memory Bandwidth Bottleneck

In [ ]:
class EducationalFeedForwardNetwork(nn.Module):
    """
    Position-wise feed-forward network with educational comments.
    
    Typically: d_model → 4×d_model → d_model
    This is often the memory bandwidth bottleneck because:
    1. Large intermediate activation (4×d_model)
    2. Two sequential matrix multiplications
    3. No data reuse (each position independent)
    """
    
    def __init__(self, d_model: int = 512, expansion_factor: int = 4, 
                 dropout: float = 0.1, activation: str = "gelu",
                 bias: bool = True):
        super().__init__()
        
        self.d_model = d_model
        self.d_ff = d_model * expansion_factor
        
        # First expansion: d_model → 4×d_model
        self.fc1 = nn.Linear(d_model, self.d_ff, bias=bias)
        
        # Second projection back: 4×d_model → d_model
        self.fc2 = nn.Linear(self.d_ff, d_model, bias=bias)
        
        self.dropout = nn.Dropout(dropout)
        
        # Activation choice matters for performance:
        # - ReLU: Simple, fast
        # - GELU: Better performance, more expensive
        # - SwiGLU: Even better but 3× parameters
        if activation == "relu":
            self.activation = F.relu
        elif activation == "gelu":
            self.activation = F.gelu
        elif activation == "silu":  # Also called Swish
            self.activation = F.silu
        else:
            raise ValueError(f"Unsupported activation: {activation}")
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of FFN.
        
        Args:
            x: [batch_size, seq_len, d_model]
            
        Returns:
            output: [batch_size, seq_len, d_model]
        """
        # First linear: O(nd²) where d_ff = 4×d_model
        hidden = self.fc1(x)  # [batch, seq_len, 4×d_model]
        
        # Activation: elementwise, cheap
        hidden = self.activation(hidden)
        
        # Dropout (training only)
        hidden = self.dropout(hidden)
        
        # Second linear: O(nd²) again
        output = self.fc2(hidden)  # [batch, seq_len, d_model]
        
        return output
    
    def flop_count(self, batch_size: int, seq_len: int) -> Dict[str, int]:
        """Calculate FLOPs for FFN forward pass."""
        # fc1: batch × seq_len × d_model × d_ff × 2
        fc1_flops = 2 * batch_size * seq_len * self.d_model * self.d_ff
        
        # Activation: approx batch × seq_len × d_ff (elementwise)
        activation_flops = batch_size * seq_len * self.d_ff
        
        # fc2: batch × seq_len × d_ff × d_model × 2
        fc2_flops = 2 * batch_size * seq_len * self.d_ff * self.d_model
        
        total = fc1_flops + activation_flops + fc2_flops
        
        return {
            'fc1': fc1_flops,
            'activation': activation_flops,
            'fc2': fc2_flops,
            'total': total
        }
    
    def memory_usage(self, batch_size: int, seq_len: int, dtype_bytes: int = 2) -> Dict[str, int]:
        """Calculate memory usage (in bytes)."""
        # Input: already allocated
        # Intermediate activation: [batch, seq_len, d_ff]
        activation_elements = batch_size * seq_len * self.d_ff
        activation_bytes = activation_elements * dtype_bytes
        
        # Output: same as input
        output_bytes = batch_size * seq_len * self.d_model * dtype_bytes
        
        return {
            'activation_mb': activation_bytes / 1024**2,
            'output_mb': output_bytes / 1024**2,
            'total_mb': (activation_bytes + output_bytes) / 1024**2
        }

## 1.3 Transformer Layer: Putting It All Together

In [ ]:
class EducationalTransformerLayer(nn.Module):
    """
    Complete transformer layer with attention and FFN.
    
    Includes:
    1. Multi-head attention
    2. Residual connection + layer normalization
    3. Feed-forward network
    4. Another residual + layer normalization
    
    Two common configurations:
    - Pre-norm: LayerNorm before sub-layer (modern, better gradient flow)
    - Post-norm: LayerNorm after sub-layer (original paper)
    """
    
    def __init__(self, d_model: int = 512, num_heads: int = 8, 
                 d_ff: int = 2048, dropout: float = 0.1,
                 activation: str = "gelu", pre_norm: bool = True):
        super().__init__()
        
        self.d_model = d_model
        self.pre_norm = pre_norm
        
        # Attention sub-layer
        self.self_attn = EducationalMultiHeadAttention(
            d_model=d_model, num_heads=num_heads, dropout=dropout, causal=True
        )
        
        # FFN sub-layer
        self.ffn = EducationalFeedForwardNetwork(
            d_model=d_model, expansion_factor=4,  # d_ff = 4×d_model
            dropout=dropout, activation=activation
        )
        
        # Layer normalizations
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Dropout for residuals
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor, padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Forward pass of transformer layer.
        
        Args:
            x: [batch_size, seq_len, d_model]
            padding_mask: [batch_size, seq_len] (True for padding)
            
        Returns:
            output: [batch_size, seq_len, d_model]
        """
        residual = x
        
        # Pre-norm or post-norm
        if self.pre_norm:
            x = self.norm1(x)
        
        # Self-attention
        attn_output, _ = self.self_attn(
            query=x, key=x, value=x,
            key_padding_mask=padding_mask
        )
        
        # Residual connection
        x = residual + self.dropout(attn_output)
        
        if not self.pre_norm:
            x = self.norm1(x)
        
        # FFN part
        residual = x
        
        if self.pre_norm:
            x = self.norm2(x)
        
        ffn_output = self.ffn(x)
        
        # Another residual connection
        x = residual + self.dropout(ffn_output)
        
        if not self.pre_norm:
            x = self.norm2(x)
        
        return x
    
    def flop_count(self, batch_size: int, seq_len: int) -> Dict[str, int]:
        """Calculate total FLOPs for this layer."""
        attn_flops = self.self_attn.flop_count(batch_size, seq_len, seq_len)
        ffn_flops = self.ffn.flop_count(batch_size, seq_len)
        
        # LayerNorm: approx 3 × batch × seq_len × d_model (mean, var, norm)
        norm_flops = 3 * batch_size * seq_len * self.d_model * 2  # 2 layer norms
        
        total = attn_flops['total'] + ffn_flops['total'] + norm_flops
        
        return {
            'attention': attn_flops['total'],
            'ffn': ffn_flops['total'],
            'layer_norm': norm_flops,
            'total': total,
            'attention_breakdown': attn_flops,
            'ffn_breakdown': ffn_flops
        }
    
    def memory_usage(self, batch_size: int, seq_len: int, dtype_bytes: int = 2) -> Dict[str, float]:
        """Calculate memory usage in MB."""
        # Attention memory
        attn_elements = batch_size * self.self_attn.num_heads * seq_len * seq_len
        attn_bytes = attn_elements * dtype_bytes  # Attention matrix
        
        # FFN memory (intermediate activation)
        ffn_memory = self.ffn.memory_usage(batch_size, seq_len, dtype_bytes)
        
        # Residuals (need to keep input for addition)
        residual_bytes = batch_size * seq_len * self.d_model * dtype_bytes
        
        total_mb = (attn_bytes + ffn_memory['total_mb'] * 1024**2 + 2 * residual_bytes) / 1024**2
        
        return {
            'attention_matrix_mb': attn_bytes / 1024**2,
            'ffn_activation_mb': ffn_memory['activation_mb'],
            'residuals_mb': 2 * residual_bytes / 1024**2,
            'total_mb': total_mb
        }

# Part 2: Memory Analysis - Understanding the Real Constraints

In [ ]:
def analyze_transformer_memory(configs):
    """
    Analyze memory usage for different transformer configurations.
    
    Configurations to test:
    1. Small: d_model=512, heads=8, seq_len=512
    2. Medium: d_model=1024, heads=16, seq_len=1024  
    3. Large: d_model=2048, heads=32, seq_len=2048
    4. GPT-3 scale: d_model=12288, heads=96, seq_len=2048
    """
    
    results = []
    
    for config in configs:
        name = config['name']
        d_model = config['d_model']
        num_heads = config['num_heads']
        seq_len = config['seq_len']
        batch_size = config.get('batch_size', 1)
        
        # Create layer
        layer = EducationalTransformerLayer(
            d_model=d_model, num_heads=num_heads
        )
        
        # Calculate FLOPs
        flops = layer.flop_count(batch_size, seq_len)
        
        # Calculate memory
        memory = layer.memory_usage(batch_size, seq_len)
        
        # Theoretical metrics
        total_params = sum(p.numel() for p in layer.parameters())
        param_memory_mb = total_params * 2 / 1024**2  # FP16
        
        results.append({
            'name': name,
            'config': config,
            'flops_g': flops['total'] / 1e9,
            'flops_breakdown': {
                'attention': flops['attention'] / 1e9,
                'ffn': flops['ffn'] / 1e9,
                'norm': flops.get('layer_norm', 0) / 1e9
            },
            'memory_mb': memory['total_mb'],
            'memory_breakdown': memory,
            'params_m': total_params / 1e6,
            'param_memory_mb': param_memory_mb,
            'total_memory_mb': memory['total_mb'] + param_memory_mb
        })
    
    return results

# Test configurations
configs = [
    {
        'name': 'Small (BERT-base-like)',
        'd_model': 768,
        'num_heads': 12,
        'seq_len': 512,
        'batch_size': 1
    },
    {
        'name': 'Medium (GPT-2 Medium-like)',
        'd_model': 1024,
        'num_heads': 16,
        'seq_len': 1024,
        'batch_size': 1
    },
    {
        'name': 'Large (GPT-3 6.7B-like)',
        'd_model': 4096,
        'num_heads': 32,
        'seq_len': 2048,
        'batch_size': 1
    },
    {
        'name': 'XL (GPT-3 175B-like)',
        'd_model': 12288,
        'num_heads': 96,
        'seq_len': 2048,
        'batch_size': 1
    }
]

print("Memory Analysis for Different Transformer Configurations")
print("=" * 80)

results = analyze_transformer_memory(configs)

for r in results:
    print(f"\n{r['name']}:")
    print(f"  Parameters: {r['params_m']:.1f}M ({r['param_memory_mb']:.1f} MB in FP16)")
    print(f"  FLOPs per layer: {r['flops_g']:.1f} GFLOPs")
    print(f"    - Attention: {r['flops_breakdown']['attention']:.1f} GFLOPs ({r['flops_breakdown']['attention']/r['flops_g']*100:.0f}%)")
    print(f"    - FFN: {r['flops_breakdown']['ffn']:.1f} GFLOPs ({r['flops_breakdown']['ffn']/r['flops_g']*100:.0f}%)")
    print(f"  Activation memory: {r['memory_mb']:.1f} MB")
    print(f"    - Attention matrix: {r['memory_breakdown']['attention_matrix_mb']:.1f} MB")
    print(f"    - FFN activation: {r['memory_breakdown']['ffn_activation_mb']:.1f} MB")
    print(f"  Total memory (params + activations): {r['total_memory_mb']:.1f} MB")
    
    # Compare with common GPU memory
    if torch.cuda.is_available():
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
        percentage = r['total_memory_mb'] / 1024 / gpu_memory * 100
        print(f"  This would use {percentage:.1f}% of a {gpu_memory:.0f}GB GPU")
        
        # How many layers could we fit?
        available_mb = gpu_memory * 1024 * 0.8  # 80% utilization
        max_layers = int(available_mb / r['total_memory_mb'])
        print(f"  Could fit ~{max_layers} such layers on this GPU")

print("\n" + "=" * 80)
print("Key Insights:")
print("1. Attention matrix memory grows with seq_len² - the n² problem!")
print("2. FFN memory grows with d_model × seq_len - often the bottleneck")
print("3. Parameters often dominate total memory for large models")
print("4. GPT-3 scale requires model parallelism - doesn't fit on one GPU")

# Part 3: Performance Profiling - Measuring Real Performance

In [ ]:
def profile_transformer_layer(layer, batch_size=1, seq_len=512, warmup=10, iterations=100):
    """
    Profile a transformer layer and measure actual performance.
    
    Returns timing breakdown and compares with theoretical limits.
    """
    
    # Move to GPU if available
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    layer = layer.to(device)
    layer.eval()
    
    # Create input
    x = torch.randn(batch_size, seq_len, layer.d_model, device=device)
    
    # Warmup
    with torch.no_grad():
        for _ in range(warmup):
            _ = layer(x)
    
    # Time forward pass
    torch.cuda.synchronize() if device.type == 'cuda' else None
    start_time = time.time()
    
    with torch.no_grad():
        for _ in range(iterations):
            _ = layer(x)
    
    torch.cuda.synchronize() if device.type == 'cuda' else None
    elapsed = time.time() - start_time
    
    avg_ms = elapsed / iterations * 1000
    
    # Calculate theoretical metrics
    flops = layer.flop_count(batch_size, seq_len)['total']
    flops_per_sec = flops * iterations / elapsed
    
    # Memory measurement
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
        _ = layer(x)
        peak_mb = torch.cuda.max_memory_allocated() / 1024**2
    else:
        peak_mb = 0
    
    # Compare with theoretical GPU performance
    if device.type == 'cuda':
        gpu_props = torch.cuda.get_device_properties(0)
        peak_tflops = gpu_props.multi_processor_count * gpu_props.clock_rate * 2 / 1e3
        
        achieved_tflops = flops_per_sec / 1e12
        utilization = achieved_tflops / peak_tflops * 100
        
        # Memory bandwidth
        # Estimated bytes transferred
        memory_bytes = peak_mb * 1024**2
        memory_bandwidth_gbs = memory_bytes / (avg_ms / 1000) / 1e9
        
        peak_bandwidth = {
            'RTX 4090': 1008,  # GB/s
            'A100': 2000,
            'H100': 3350
        }.get(gpu_props.name, 500)  # Default estimate
        
        bandwidth_utilization = memory_bandwidth_gbs / peak_bandwidth * 100
    else:
        achieved_tflops = utilization = memory_bandwidth_gbs = bandwidth_utilization = 0
    
    return {
        'avg_ms': avg_ms,
        'flops': flops,
        'flops_per_sec': flops_per_sec,
        'achieved_tflops': achieved_tflops,
        'utilization_percent': utilization,
        'peak_memory_mb': peak_mb,
        'memory_bandwidth_gbs': memory_bandwidth_gbs,
        'bandwidth_utilization_percent': bandwidth_utilization,
        'tokens_per_second': (batch_size * seq_len) / (avg_ms / 1000)
    }

# Profile different sequence lengths
print("Performance Profiling: Sequence Length Scaling")
print("=" * 80)

seq_lengths = [128, 256, 512, 1024, 2048]
results = []

for seq_len in seq_lengths:
    layer = EducationalTransformerLayer(d_model=512, num_heads=8)
    
    try:
        profile = profile_transformer_layer(layer, seq_len=seq_len, iterations=50)
        results.append((seq_len, profile))
        
        print(f"\nSequence Length: {seq_len}")
        print(f"  Latency: {profile['avg_ms']:.2f} ms")
        print(f"  Tokens/sec: {profile['tokens_per_second']:.0f}")
        print(f"  FLOPs: {profile['flops'] / 1e9:.1f} GFLOPs")
        
        if torch.cuda.is_available():
            print(f"  Achieved: {profile['achieved_tflops']:.2f} TFLOPS ({profile['utilization_percent']:.1f}% of peak)")
            print(f"  Memory: {profile['peak_memory_mb']:.1f} MB")
            print(f"  Bandwidth: {profile['memory_bandwidth_gbs']:.1f} GB/s ({profile['bandwidth_utilization_percent']:.1f}% of peak)")
            
    except RuntimeError as e:
        print(f"\nSequence Length {seq_len}: FAILED - {str(e)}")
        break

# Plot results
if results:
    seq_lens = [r[0] for r in results]
    latencies = [r[1]['avg_ms'] for r in results]
    tokens_per_sec = [r[1]['tokens_per_second'] for r in results]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Latency plot
    ax1.plot(seq_lens, latencies, 'o-', linewidth=2, markersize=8)
    ax1.set_xlabel('Sequence Length')
    ax1.set_ylabel('Latency (ms)')
    ax1.set_title('Latency vs Sequence Length')
    ax1.grid(True, alpha=0.3)
    
    # Fit quadratic to show O(n²) scaling
    if len(seq_lens) >= 3:
        coeffs = np.polyfit(seq_lens, latencies, 2)
        poly = np.poly1d(coeffs)
        x_fit = np.linspace(min(seq_lens), max(seq_lens), 100)
        ax1.plot(x_fit, poly(x_fit), '--', alpha=0.5, label=f'O(n²) fit')
        ax1.legend()
    
    # Throughput plot
    ax2.plot(seq_lens, tokens_per_sec, 'o-', linewidth=2, markersize=8, color='orange')
    ax2.set_xlabel('Sequence Length')
    ax2.set_ylabel('Tokens per Second')
    ax2.set_title('Throughput vs Sequence Length')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "=" * 80)
    print("Key Insights:")
    print("1. Latency grows quadratically with sequence length (O(n²) attention)")
    print("2. Throughput peaks at some sequence length (memory vs compute trade-off)")
    print("3. Memory bandwidth often becomes the bottleneck for long sequences")

# Part 4: Architectural Experiments - Understanding Trade-offs

In [ ]:
def experiment_head_count():
    """
    Experiment with different numbers of attention heads.
    
    Questions:
    1. How does head count affect performance?
    2. What's the optimal head dimension?
    3. Is there a memory vs speed trade-off?
    """
    
    d_model = 512
    head_counts = [1, 2, 4, 8, 16, 32]
    
    print("Experiment: Head Count vs Performance")
    print("=" * 80)
    print(f"Fixed d_model: {d_model}")
    print()
    
    results = []
    
    for num_heads in head_counts:
        # Check divisibility
        if d_model % num_heads != 0:
            print(f"  {num_heads} heads: Skipped (not divisible)")
            continue
            
        head_dim = d_model // num_heads
        
        # Create layer
        layer = EducationalTransformerLayer(d_model=d_model, num_heads=num_heads)
        
        # Profile
        try:
            profile = profile_transformer_layer(layer, seq_len=256, iterations=30)
            
            # Calculate theoretical metrics
            flops = layer.flop_count(1, 256)
            memory = layer.memory_usage(1, 256)
            
            results.append({
                'num_heads': num_heads,
                'head_dim': head_dim,
                'latency_ms': profile['avg_ms'],
                'tokens_per_sec': profile['tokens_per_second'],
                'flops_g': flops['total'] / 1e9,
                'memory_mb': memory['total_mb'],
                'attention_memory_mb': memory['attention_matrix_mb']
            })
            
            print(f"  {num_heads:2d} heads (dim={head_dim:3d}): {profile['avg_ms']:5.2f} ms, "
                  f"{profile['tokens_per_second']:6.0f} tokens/sec, "
                  f"{memory['attention_matrix_mb']:5.1f} MB attention matrix")
                  
        except RuntimeError as e:
            print(f"  {num_heads} heads: FAILED - {str(e)}")
    
    return results

def experiment_model_dimension():
    """
    Experiment with different model dimensions.
    
    Questions:
    1. How does d_model affect performance?
    2. What's the compute vs memory trade-off?
    3. How does FLOP scaling compare to actual speed?
    """
    
    d_models = [256, 512, 768, 1024, 1536, 2048]
    heads_per_dim = 8  # Keep heads proportional
    
    print("\n" + "=" * 80)
    print("Experiment: Model Dimension vs Performance")
    print("=" * 80)
    print(f"Heads per dimension: {heads_per_dim}")
    print()
    
    results = []
    
    for d_model in d_models:
        num_heads = max(1, d_model // heads_per_dim)
        
        # Ensure divisibility
        while d_model % num_heads != 0 and num_heads > 1:
            num_heads -= 1
        
        if num_heads == 0:
            continue
            
        # Create layer
        layer = EducationalTransformerLayer(d_model=d_model, num_heads=num_heads)
        
        # Profile with smaller sequence for large models
        seq_len = 128 if d_model >= 1536 else 256
        
        try:
            profile = profile_transformer_layer(layer, seq_len=seq_len, iterations=20)
            
            # Calculate theoretical metrics
            flops = layer.flop_count(1, seq_len)
            memory = layer.memory_usage(1, seq_len)
            
            results.append({
                'd_model': d_model,
                'num_heads': num_heads,
                'head_dim': d_model // num_heads,
                'latency_ms': profile['avg_ms'],
                'tokens_per_sec': profile['tokens_per_second'],
                'flops_g': flops['total'] / 1e9,
                'memory_mb': memory['total_mb'],
                'flops_per_ms': (flops['total'] / 1e9) / (profile['avg_ms'] / 1000)  # GFLOPs/sec
            })
            
            print(f"  d_model={d_model:4d}, heads={num_heads:2d}: {profile['avg_ms']:6.2f} ms, "
                  f"{flops['total']/1e9:6.1f} GFLOPs, {memory['total_mb']:6.1f} MB, "
                  f"{results[-1]['flops_per_ms']:6.1f} GFLOPs/sec")
                  
        except RuntimeError as e:
            print(f"  d_model={d_model}: FAILED - {str(e)}")
            break
    
    return results

# Run experiments
head_results = experiment_head_count()
dim_results = experiment_model_dimension()

# Plot results
if head_results:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    heads = [r['num_heads'] for r in head_results]
    latencies = [r['latency_ms'] for r in head_results]
    memory = [r['attention_memory_mb'] for r in head_results]
    
    ax1.plot(heads, latencies, 'o-', linewidth=2, markersize=8)
    ax1.set_xlabel('Number of Heads')
    ax1.set_ylabel('Latency (ms)')
    ax1.set_title('Head Count vs Latency')
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(heads, memory, 'o-', linewidth=2, markersize=8, color='orange')
    ax2.set_xlabel('Number of Heads')
    ax2.set_ylabel('Attention Matrix Memory (MB)')
    ax2.set_title('Head Count vs Memory')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

if dim_results:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    d_models = [r['d_model'] for r in dim_results]
    latencies = [r['latency_ms'] for r in dim_results]
    flops_per_sec = [r['flops_per_ms'] * 1000 for r in dim_results]  # Convert to GFLOPs/sec
    
    ax1.plot(d_models, latencies, 'o-', linewidth=2, markersize=8)
    ax1.set_xlabel('Model Dimension (d_model)')
    ax1.set_ylabel('Latency (ms)')
    ax1.set_title('Model Size vs Latency')
    ax1.grid(True, alpha=0.3)
    
    # Fit to show scaling
    if len(d_models) >= 3:
        # Should be roughly O(d²)
        coeffs = np.polyfit(d_models, latencies, 2)
        poly = np.poly1d(coeffs)
        x_fit = np.linspace(min(d_models), max(d_models), 100)
        ax1.plot(x_fit, poly(x_fit), '--', alpha=0.5, label='O(d²) fit')
        ax1.legend()
    
    ax2.plot(d_models, flops_per_sec, 'o-', linewidth=2, markersize=8, color='green')
    ax2.set_xlabel('Model Dimension (d_model)')
    ax2.set_ylabel('Compute Efficiency (GFLOPs/sec)')
    ax2.set_title('Model Size vs Compute Efficiency')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "=" * 80)
    print("Key Insights from Experiments:")
    print("1. More heads = more parallelization but more memory overhead")
    print("2. There's an optimal head count for given d_model (balance heads vs head_dim)")
    print("3. Model dimension affects latency quadratically (O(d²) from linear layers)")
    print("4. Larger models achieve higher FLOP/s (better GPU utilization)")
    print("5. Memory grows faster than compute requirements")

# Part 5: Optimization Exploration - Simple Optimizations That Help

In [ ]:
def explore_optimizations():
    """
    Explore simple optimizations and measure their impact.
    
    Optimizations to try:
    1. Mixed precision (FP16)
    2. Kernel fusion (simulated)
    3. Memory layout optimizations
    4. Batch size optimization
    """
    
    if not torch.cuda.is_available():
        print("CUDA not available - skipping optimization experiments")
        return
    
    print("Optimization Experiments")
    print("=" * 80)
    
    # Baseline model
    layer = EducationalTransformerLayer(d_model=512, num_heads=8).cuda()
    layer.eval()
    
    x = torch.randn(1, 512, 512, device='cuda')
    
    # 1. Baseline (FP32)
    with torch.no_grad():
        torch.cuda.synchronize()
        start = time.time()
        for _ in range(100):
            _ = layer(x)
        torch.cuda.synchronize()
        fp32_time = time.time() - start
    
    # 2. FP16
    layer_fp16 = EducationalTransformerLayer(d_model=512, num_heads=8).cuda().half()
    layer_fp16.eval()
    x_fp16 = x.half()
    
    with torch.no_grad():
        torch.cuda.synchronize()
        start = time.time()
        for _ in range(100):
            _ = layer_fp16(x_fp16)
        torch.cuda.synchronize()
        fp16_time = time.time() - start
    
    # 3. Batch size optimization
    batch_sizes = [1, 2, 4, 8, 16]
    batch_results = []
    
    for bs in batch_sizes:
        x_batch = torch.randn(bs, 512, 512, device='cuda').half()
        
        with torch.no_grad():
            torch.cuda.synchronize()
            start = time.time()
            for _ in range(100 // bs):  # Keep total tokens constant
                _ = layer_fp16(x_batch)
            torch.cuda.synchronize()
            elapsed = time.time() - start
        
        tokens_per_sec = (bs * 512 * (100 // bs)) / elapsed
        batch_results.append((bs, tokens_per_sec))
    
    # 4. Memory layout experiment
    # Compare contiguous vs non-contiguous memory
    size = 1024 * 1024 * 100  # 100MB
    
    contiguous = torch.randn(size, device='cuda')
    non_contiguous = contiguous[::2]  # Strided access
    
    # Time sequential access
    torch.cuda.synchronize()
    start = time.time()
    acc = 0
    for i in range(0, size, 1024):
        acc += contiguous[i:i+1024].sum()
    torch.cuda.synchronize()
    contiguous_time = time.time() - start
    
    # Time strided access
    torch.cuda.synchronize()
    start = time.time()
    acc = 0
    for i in range(0, size//2, 512):
        acc += non_contiguous[i:i+512].sum()
    torch.cuda.synchronize()
    non_contiguous_time = time.time() - start
    
    # Print results
    print(f"\n1. Precision Optimization:")
    print(f"   FP32: {fp32_time:.3f}s ({100/fp32_time:.1f} iterations/sec)")
    print(f"   FP16: {fp16_time:.3f}s ({100/fp16_time:.1f} iterations/sec)")
    print(f"   Speedup: {fp32_time/fp16_time:.2f}x")
    
    print(f"\n2. Batch Size Optimization:")
    for bs, tps in batch_results:
        print(f"   Batch {bs:2d}: {tps:.0f} tokens/sec")
    
    # Find optimal batch size
    optimal_bs, max_tps = max(batch_results, key=lambda x: x[1])
    print(f"   Optimal batch size: {optimal_bs} ({max_tps:.0f} tokens/sec)")
    
    print(f"\n3. Memory Layout Optimization:")
    print(f"   Contiguous access: {contiguous_time:.3f}s")
    print(f"   Strided access:    {non_contiguous_time:.3f}s")
    print(f"   Slowdown: {non_contiguous_time/contiguous_time:.2f}x")
    
    print(f"\n" + "=" * 80)
    print("Optimization Takeaways:")
    print("1. FP16 gives ~2x speedup with modern GPUs (tensor cores)")
    print("2. Optimal batch size maximizes throughput within memory limits")
    print("3. Memory access patterns dramatically affect performance")
    print("4. Real optimizations combine multiple techniques")
    
    # Plot batch size results
    if batch_results:
        plt.figure(figsize=(8, 4))
        
        batch_sizes = [r[0] for r in batch_results]
        tokens_per_sec = [r[1] for r in batch_results]
        
        plt.plot(batch_sizes, tokens_per_sec, 'o-', linewidth=2, markersize=8)
        plt.xlabel('Batch Size')
        plt.ylabel('Tokens per Second')
        plt.title('Batch Size vs Throughput')
        plt.grid(True, alpha=0.3)
        
        # Mark optimal point
        plt.scatter([optimal_bs], [max_tps], color='red', s=100, zorder=5)
        plt.text(optimal_bs, max_tps, f'Optimal\n{optimal_bs} batch', 
                 ha='center', va='bottom', fontweight='bold')
        
        plt.tight_layout()
        plt.show()

# Run optimization experiments
explore_optimizations()

# 🎯 Lab Completion Checklist

## ✅ What You Should Have Learned

### 1. Implementation Understanding
- [ ] Can implement multi-head attention from memory
- [ ] Understand tensor shapes at each step
- [ ] Can explain the purpose of each operation

### 2. Performance Analysis
- [ ] Can calculate FLOPs for a given configuration
- [ ] Can estimate memory usage
- [ ] Can profile and identify bottlenecks

### 3. Architectural Trade-offs
- [ ] Understand head count vs head dimension trade-off
- [ ] Know how sequence length affects performance
- [ ] Can explain why model dimension affects latency quadratically

### 4. Optimization Awareness
- [ ] Know the benefits of mixed precision
- [ ] Understand batch size optimization
- [ ] Appreciate importance of memory access patterns

## 📊 Key Metrics to Remember

### For transformer layer with d_model=512, heads=8, seq_len=512:
- **FLOPs**: ~2-3 GFLOPs per forward pass
- **Memory**: ~50-100 MB for activations
- **Latency**: ~5-20 ms on modern GPU
- **Memory bound** for attention, **compute bound** for linear layers

## 🔮 What's Next?

### In Week 2, you'll learn:
1. **KV Caching**: How to avoid recomputing attention for previous tokens
2. **FlashAttention**: Memory-efficient attention implementation
3. **Batch Size Optimization**: Finding the sweet spot for your hardware
4. **Kernel Fusion**: Combining operations for better performance

### How This Lab Prepares You:
- Understanding attention complexity → optimizing with KV caching
- Profiling skills → identifying where FlashAttention helps
- Memory analysis → optimizing batch size
- Implementation knowledge → understanding kernel fusion

## 📝 Exercise: Self-Assessment

Test your understanding by answering:

1. **Why** does attention have O(n²) complexity?
2. **What** is the memory bottleneck in FFN layers?
3. **How** does increasing batch size affect throughput and latency?
4. **When** would you choose more heads vs larger head dimension?
5. **What** optimization would you try first for a memory-bound model?

## 🚀 Challenge Problems (Optional)

For extra credit:

1. **Implement memory-efficient attention** that doesn't materialize the full n×n matrix
2. **Add gradient checkpointing** to reduce memory during training
3. **Implement fused operations** (LayerNorm + residual in one kernel)
4. **Profile on different hardware** (CPU, different GPUs) and compare
5. **Extend to multi-layer transformer** and measure scaling

---

**Congratulations on completing Lab 1.1!** You now have a solid foundation for understanding and optimizing transformer inference. Save this notebook with your results and proceed to Lab 1.2.